In [1]:
import json
import time

# External Dependencies:
import boto3  # AWS SDK for Python
from botocore.exceptions import ClientError  # for error handling
from IPython.display import Markdown, display  # Notebook display utilities

In [3]:
session = boto3.Session()
region = session.region_name
bedrock = boto3.client(service_name="bedrock-runtime", region_name=region)

In [19]:
MODELS = {
    "Claude Haiku 4.5": "eu.anthropic.claude-haiku-4-5-20251001-v1:0",
    "Claude Sonnet 4.5": "eu.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "Amazon Nova Pro": "eu.amazon.nova-pro-v1:0",
    "Amazon Nova 2 Lite": "eu.amazon.nova-2-lite-v1:0",
    "Meta Llama 3.1 70B Instruct": "eu.meta.llama3-1-70b-instruct-v1:0",
}

In [20]:
def display_response(response, model_name=None):
    if model_name:
        display(Markdown(f"### Response from {model_name}"))
    display(Markdown(response))
    print("\n" + "-" * 80 + "\n")

In [21]:
print(bedrock.meta.region_name)

eu-north-1


In [22]:
#text summarisation

In [23]:
text_to_summarize = """
AWS took all of that feedback from customers, and today we are excited to announce Amazon Bedrock, \
a new service that makes FMs from AI21 Labs, Anthropic, Stability AI, and Amazon accessible via an API. \
Bedrock is the easiest way for customers to build and scale generative AI-based applications using FMs, \
democratizing access for all builders. Bedrock will offer the ability to access a range of powerful FMs \
for text and images—including Amazons Titan FMs, which consist of two new LLMs we're also announcing \
today—through a scalable, reliable, and secure AWS managed service. With Bedrock's serverless experience, \
customers can easily find the right model for what they're trying to get done, get started quickly, privately \
customize FMs with their own data, and easily integrate and deploy them into their applications using the AWS \
tools and capabilities they are familiar with, without having to manage any infrastructure (including integrations \
with Amazon SageMaker ML features like Experiments to test different models and Pipelines to manage their FMs at scale).
"""

In [24]:
# Create prompt for summarization (format for Claude Haiku 4.5
prompt = f"""Please provide a summary of the following text. Do not add any information that is not mentioned in the text below.
<text>
{text_to_summarize}
</text>
"""

In [25]:
# Create request body for Claude Haiku 4.5
claude_body = json.dumps(
    {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "temperature": 0.5,
        "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
    }
)

In [26]:
# Send request to Claude Haiku 4.5
try:
    response = bedrock.invoke_model(
        modelId=MODELS["Claude Haiku 4.5"],
        body=claude_body,
        accept="application/json",
        contentType="application/json",
    )
    response_body = json.loads(response.get("body").read())

    # Extract and display the response text
    claude_summary = response_body["content"][0]["text"]
    display_response(claude_summary, "Claude Haiku 4.5 (Invoke Model API)")

except ClientError as error:
    if error.response["Error"]["Code"] == "AccessDeniedException":
        print(
            f"\x1b[41m{error.response['Error']['Message']}\
            \nTo troubleshoot this issue please refer to the following resources.\
            \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
            \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n"
        )
    else:
        raise error

### Response from Claude Haiku 4.5 (Invoke Model API)

# Summary

AWS announced Amazon Bedrock, a new service that provides API access to foundation models (FMs) from AI21 Labs, Anthropic, Stability AI, and Amazon. Bedrock aims to make it easier for customers to build and scale generative AI applications by democratizing access to FMs. The service offers a serverless, managed AWS experience that allows customers to access various FMs for text and images, including Amazon's new Titan FMs (two new large language models). Through Bedrock, users can select appropriate models, customize them privately with their own data, and integrate them into applications using familiar AWS tools without managing infrastructure. The service includes integrations with Amazon SageMaker features like Experiments and Pipelines.


--------------------------------------------------------------------------------



In [ ]:
#converse request

In [46]:
converse_request = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": f"Please provide a concise summary of the following text in 2-3 sentences. Text to summarize: {text_to_summarize}"
                }
            ],
        }
    ],
    "inferenceConfig": {
        "maxTokens": 500,
        "temperature": 0.4,
    },
}

# Call Claude Haiku 4.5 with Converse API
try:
    response = bedrock.converse(
        modelId=MODELS["Amazon Nova Pro"],
        messages=converse_request["messages"],
        inferenceConfig=converse_request["inferenceConfig"],
    )

    # Extract the model's response
    claude_converse_response = response["output"]["message"]["content"][0]["text"]
    display_response(claude_converse_response, "Claude Haiku 4.5 (Converse API)")
except ClientError as error:
    if error.response["Error"]["Code"] == "AccessDeniedException":
        print(
            f"\x1b[41m{error.response['Error']['Code']}: {error.response['Error']['Message']}\x1b[0m"
        )
        print("Please ensure you have the necessary permissions for Amazon Bedrock.")
    else:
        raise error

### Response from Claude Haiku 4.5 (Converse API)

AWS has launched Amazon Bedrock, a new service providing easy access to Foundation Models (FMs) from various providers like AI21 Labs, Anthropic, Stability AI, and Amazon through an API. Bedrock aims to democratize generative AI application development by offering scalable, secure, and managed access to powerful text and image FMs, including Amazon's new Titan FMs. It features a serverless experience allowing customers to quickly select, customize, and deploy models using familiar AWS tools without infrastructure management.


--------------------------------------------------------------------------------



In [33]:
import sagemaker
print(sagemaker.__file__)

/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py


In [37]:
import sagemaker
print(sagemaker.__version__)
print(sagemaker.__file__)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 import sagemaker                                                                             │
│ ❱ 2 print(sagemaker.__version__)                                                                 │
│   3 print(sagemaker.__file__)                                                                    │
│   4                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
AttributeError: module 'sagemaker' has no attribute '__version__'

In [39]:
import importlib.metadata
print(importlib.metadata.version("sagemaker"))

3.21.0


In [38]:
converse_request = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": f"Please provide a concise summary of the following text in 2-3 sentences. Text to summarize: {text_to_summarize}"
                }
            ],
        }
    ],
    "inferenceConfig": {
        "maxTokens": 500,
        "temperature": 0.4,
    },
}

# Call Claude Haiku 4.5 with Converse API
try:
    response = bedrock.converse(
        modelId=MODELS["Claude Haiku 4.5"],
        messages=converse_request["messages"],
        inferenceConfig=converse_request["inferenceConfig"],
    )

    # Extract the model's response
    claude_converse_response = response["output"]["message"]["content"][0]["text"]
    display_response(claude_converse_response, "Claude Haiku 4.5 (Converse API)")
except ClientError as error:
    if error.response["Error"]["Code"] == "AccessDeniedException":
        print(
            f"\x1b[41m{error.response['Error']['Code']}: {error.response['Error']['Message']}\x1b[0m"
        )
        print("Please ensure you have the necessary permissions for Amazon Bedrock.")
    else:
        raise error

AccessDeniedException: Model access is denied due to IAM user or service role is not authorized to perform the required AWS Marketplace actions (aws-marketplace:ViewSubscriptions, aws-marketplace:Subscribe) to enable access to this model. Refer to the Amazon Bedrock documentation for further details. Your AWS Marketplace subscription for this model cannot be completed at this time. If you recently fixed this issue, try again after 2 minutes.
Please ensure you have the necessary permissions for Amazon Bedrock.


In [43]:
import boto3
iam = boto3.client("iam")
role_name = "AmazonSageMakerAdminIAMExecutionRole"

print("Inline policies:", iam.list_role_policies(RoleName=role_name)["PolicyNames"])
print("Attached managed policies:", [p["PolicyName"] for p in iam.list_attached_role_policies(RoleName=role_name)["AttachedPolicies"]])

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:5                                                                                    │
│                                                                                                  │
│   2 iam = boto3.client("iam")                                                                    │
│   3 role_name = "AmazonSageMakerAdminIAMExecutionRole"                                           │
│   4                                                                                              │
│ ❱ 5 print("Inline policies:", iam.list_role_policies(RoleName=role_name)["PolicyNames"])         │
│   6 print("Attached managed policies:", [p["PolicyName"] for p in iam.list_attached_role_pol     │
│   7                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:606 in _api_call                      │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    609                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1094 in _make_api_call                │
│                                                                                                  │
│   1091 │   │   │   │   'error_code_override'                                                     │
│   1092 │   │   │   ) or error_info.get("Code")                                                   │
│   1093 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1094 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1095 │   │   else:                                                                             │
│   1096 │   │   │   return parsed_response                                                        │
│   1097                                                                                           │
╰────────────────────────────────────────────────────────────

In [40]:
!pip show sagemaker

Name: sagemaker
Version: 3.21.0
Summary: Open source library for training and deploying models on Amazon SageMaker.
Home-page: https://github.com/aws/sagemaker-python-sdk
Author: Amazon Web Services
Author-email: 
License: 
Location: /opt/conda/lib/python3.12/site-packages
Requires: sagemaker-core, sagemaker-mlops, sagemaker-serve, sagemaker-train
Required-by: 


In [41]:
import boto3
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(identity["Arn"])

arn:aws:sts::415039714180:assumed-role/AmazonSageMakerAdminIAMExecutionRole/SageMaker
